In [1]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter

In [3]:
df_true = pd.read_csv("../True.csv")
df_fake = pd.read_csv("../Fake.csv")

df_true['label'] = 1
df_fake['label'] = 0

df = pd.concat([df_true, df_fake], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df['content'] = df['title'] + ' ' + df['text']
df = df[['content', 'label']]

In [4]:
df

,content,label
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,0
1,Failed GOP Candidates Remembered In Hilarious...,0
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,0
3,California AG pledges to defend birth control ...,1
4,AZ RANCHERS Living On US-Mexico Border Destroy...,0
...,...,...
44893,Nigeria says U.S. agrees delayed $593 million ...,1
44894,Boiler Room #62 – Fatal Illusions Tune in to t...,0
44895,ATHEISTS SUE GOVERNOR OF TEXAS Over Display on...,0
44896,Republican tax plan would deal financial hit t...,1


In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub('<.*?>', ' ', text)
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = re.sub('\s+', ' ', text).strip()
    return text

df['content'] = df['content'].apply(clean_text)

In [ ]:
df['tokens'] = df['content'].apply(lambda x: x.split())

all_words = [word for tokens in df['tokens'] for word in tokens]
word_counts = Counter(all_words)

vocab_size = 20000
vocab = ['<PAD>', '<UNK>'] + [word for word, count in word_counts.most_common(vocab_size)]
word2idx = {word: idx for idx, word in enumerate(vocab)}